# W6_02 · From Neighbour Messages to a New Link
Explore 20 nodes → train GraphSAGE → follow a prediction → reveal suggested links.

Run top to bottom. **No API key or GPU needed.**

## 1. Setup
Click **▶** on the setup cell, wait for **Ready**, then run the cells below.
Everything needed for the exercise, including the graph sample, is built into this notebook.
The setup code is folded so you can focus on the short exercise cells.

In [ ]:
#@title 1. Setup — click ▶ to start
%pip -q install torch_geometric==2.8.0 networkx numpy

"""Visual teaching helpers for W6_02, embedded by the notebook generator.

The small, connected class-graph sample is used for both training and viewing.
GraphSAGE learns node representations; an inner-product decoder ranks absent
undirected links. Replay actual messages and updates before decoding a new link.
"""
import hashlib
import html
import json
import random
import re
from collections import Counter

import networkx as nx
import numpy as np
import torch
from IPython.display import HTML
from torch_geometric.nn.models import GAE, GraphSAGE


COLORS = {'Material': '#2463b4', 'Property': '#8c54a2', 'Process': '#b55b20',
          'Condition': '#637333', 'Electrolyte': '#147d83', 'Cell': '#b44b68'}


class GraphSAGELab:
    def __init__(self, graph, max_nodes=20):
        full = nx.Graph()
        full.add_nodes_from((n['id'], n) for n in graph['nodes'] if n['type'] != 'Paper')
        full.add_edges_from((e['source'], e['target']) for e in graph['edges']
                            if e['source'] in full and e['target'] in full and e['source'] != e['target'])
        if not full.edges:
            raise ValueError('The graph needs at least one link between non-Paper nodes.')
        seed = max((n for n in full if full.degree[n]),
                   key=lambda n: (full.nodes[n]['type'] == 'Material', full.degree[n], n))
        selected, counts = [seed], Counter([full.nodes[seed]['type']])
        boundary = set(full.neighbors(seed))
        while boundary and len(selected) < max_nodes:
            n = min(boundary, key=lambda n: (counts[full.nodes[n]['type']], -full.degree[n], n))
            selected.append(n)
            counts[full.nodes[n]['type']] += 1
            boundary.update(full.neighbors(n))
            boundary.difference_update(selected)
        self.nodes = [dict(full.nodes[n]) for n in selected]
        self.graph = nx.relabel_nodes(full.subgraph(selected), {n: i for i, n in enumerate(selected)}).copy()
        self.known = {tuple(sorted((a, b))) for a, b in self.graph.edges}
        self.types = sorted({n['type'] for n in self.nodes})
        self.positions = {int(i): (round(float(x) * 430 + 500, 2), round(float(y) * 225 + 275, 2))
                          for i, (x, y) in nx.spring_layout(self.graph, seed=42, iterations=150).items()}
        features = np.zeros((len(self.nodes), len(self.types) + 64), dtype=np.float32)
        for i, node in enumerate(self.nodes):
            features[i, self.types.index(node['type'])] = 1
            for word in re.findall(r'\w+', node['label'].lower()):
                bucket = int(hashlib.sha256(word.encode()).hexdigest()[:8], 16) % 64
                features[i, len(self.types) + bucket] += 1
        self.features = torch.from_numpy(features)
        self.features /= self.features.norm(dim=1, keepdim=True).clamp_min(1e-9)
        pairs = torch.tensor(sorted(self.known), dtype=torch.long).T.contiguous()
        self.edge_index = torch.cat((pairs, pairs.flip(0)), dim=1)
        self.embeddings = None
        self.traces = []
        self.message_trace = []

    def fit(self, layers=2, epochs=40):
        if type(layers) is not int or not 1 <= layers <= 3:
            raise ValueError('Set layers to 1, 2 or 3.')
        if type(epochs) is not int or not 1 <= epochs <= 200:
            raise ValueError('Set epochs to a whole number from 1 to 200.')
        if len(self.known) == len(self.nodes) * (len(self.nodes) - 1) // 2:
            raise ValueError('This graph already contains every possible link. Use a larger graph.')
        random.seed(42)
        np.random.seed(42)
        torch.manual_seed(42)
        self.model = GAE(GraphSAGE(self.features.shape[1], 32, num_layers=layers, out_channels=16))
        optimizer = torch.optim.Adam(self.model.parameters(), lr=0.01)
        old_threads = torch.get_num_threads()
        torch.set_num_threads(min(old_threads, 2))
        try:
            self.model.train()
            for _ in range(epochs):
                optimizer.zero_grad()
                z = self.model.encode(self.features, self.edge_index)
                loss = self.model.recon_loss(z, self.edge_index)
                loss.backward()
                optimizer.step()
            self.model.eval()
            with torch.no_grad():
                self.embeddings = self.model.encode(self.features, self.edge_index).detach()
                self.traces = [self.features]
                self.message_trace = []
                z = self.features
                for i, layer in enumerate(self.model.encoder.convs):
                    mean = torch.stack([z[list(self.graph.neighbors(n))].mean(0) for n in range(len(self.nodes))])
                    own, neighbours = layer.lin_r(z), layer.lin_l(mean)
                    updated = layer(z, self.edge_index)
                    if i < layers - 1:
                        updated = updated.relu()
                    self.message_trace.append({'input': z.detach(), 'mean': mean.detach(),
                        'self_term': own.detach(), 'neighbour_term': neighbours.detach(), 'output': updated.detach()})
                    z = updated
                    self.traces.append(z.detach())
        finally:
            torch.set_num_threads(old_threads)
        return self

    def predict(self, source_type='Material', target_type='Property', top_k=3):
        if source_type not in self.types or target_type not in self.types:
            raise ValueError('Choose node types from: ' + ', '.join(self.types))
        if type(top_k) is not int or not 0 <= top_k <= 50:
            raise ValueError('Set top_k to a whole number from 0 to 50.')
        if self.embeddings is None:
            raise ValueError('Run the GraphSAGE cell before predicting links.')
        candidates, seen = [], set()
        for a, left in enumerate(self.nodes):
            if left['type'] != source_type:
                continue
            for b, right in enumerate(self.nodes):
                pair = tuple(sorted((a, b)))
                if right['type'] != target_type or a == b or pair in self.known or pair in seen:
                    continue
                seen.add(pair)
                score = float(torch.dot(self.embeddings[a], self.embeddings[b]))
                candidates.append({'source': a, 'target': b, 'score': score})
        return sorted(candidates, key=lambda r: (-r['score'], r['source'], r['target']))[:top_k]

    def before(self):
        return self._display('before')

    def walkthrough(self):
        return self.explain_link()

    def explain_link(self, source_type='Material', target_type='Property', top_k=3):
        predictions = self.predict(source_type, target_type, top_k)
        if not predictions:
            raise ValueError('No absent pairs for this setting. Choose another node-type pair.')
        return self._display('explain', predictions, f'{source_type} — {target_type}', top_k)

    def after(self, source_type='Material', target_type='Property', top_k=3):
        predictions = self.predict(source_type, target_type, top_k)
        return self._display('after', predictions, f'{source_type} — {target_type}', top_k)

    def _display(self, mode, predictions=None, pair_label='', requested=0):
        document = self.document(mode, predictions, pair_label, requested)
        height = 1240 if mode in ('explain', 'walkthrough') else 880
        return HTML(f'<div><iframe title="GraphSAGE interactive graph" style="width:100%;height:{height}px;'
                    'border:0" sandbox="allow-scripts allow-same-origin" srcdoc="'
                    + html.escape(document, quote=True) + '"></iframe></div>')

    def document(self, mode='before', predictions=None, pair_label='', requested=0):
        """Return the same standalone HTML used in Colab, for offline review."""
        if mode == 'walkthrough':
            mode = 'explain'
            predictions = self.predict()
        if mode == 'explain' and predictions is None:
            predictions = self.predict()
        predictions = [] if mode == 'before' else (predictions or [])
        checked, seen = [], set()
        for prediction in predictions:
            a, b = prediction['source'], prediction['target']
            pair = tuple(sorted((a, b)))
            if a == b or pair in self.known:
                raise ValueError('A suggested link must not be an existing link or a self-link.')
            if pair in seen:
                raise ValueError('Duplicate suggested link.')
            seen.add(pair)
            products = self.embeddings[a] * self.embeddings[b]
            checked.append({**prediction, 'score': float(products.sum()),
                'contributions': products.tolist(), 'link_score': float(products.sum().sigmoid())})
        payload = {
            'mode': mode, 'pairLabel': pair_label, 'requested': requested,
            'nodes': [dict(id=i, label=n['label'], type=n['type'], color=COLORS.get(n['type'], '#526373'),
                           x=self.positions[i][0], y=self.positions[i][1]) for i, n in enumerate(self.nodes)],
            'edges': sorted(self.known), 'predictions': checked,
            'vectors': [t[:, :8].tolist() for t in self.traces],
            'layers': [{k: v[:, :4].tolist() for k, v in state.items()} for state in self.message_trace],
            'types': {t: COLORS.get(t, '#526373') for t in self.types},
        }
        data = json.dumps(payload, ensure_ascii=True).replace('<', '\\u003c')
        return VIEWER.replace('__PAYLOAD__', data)


VIEWER = r'''<!doctype html><html lang="en"><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>GraphSAGE visual lab</title>
<style>
*{box-sizing:border-box}body{margin:0;padding:18px;font:15px/1.5 system-ui,sans-serif;color:#172b42;background:#fff}
h2{font-size:21px;margin:0 0 5px}p{margin:5px 0 12px}.muted{color:#526373}.toolbar{display:flex;gap:9px;align-items:center;flex-wrap:wrap;margin:10px 0}
button,select{font:inherit;border:1px solid #bac6d3;background:#fff;color:#172b42;border-radius:5px;min-height:44px;padding:6px 12px;cursor:pointer}
select{max-width:100%;width:310px}button:hover{background:#edf4fc}button:disabled{opacity:.5;cursor:default}
:focus-visible{outline:3px solid #2463b4;outline-offset:3px}.legend{display:flex;flex-wrap:wrap;gap:8px 18px;font-size:13px}
.dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:6px}.line{display:inline-block;width:22px;border-top:2px solid #bac2cb;margin:0 6px 3px 0}
.dash{border-color:#b84d13;border-top-style:dashed}svg{display:block;width:100%;height:440px;border:1px solid #d4dce6;background:#fafcff;touch-action:none;border-radius:6px}
svg circle{cursor:pointer}svg text{pointer-events:none;font-family:system-ui;font-size:12px;paint-order:stroke;stroke:#fff;stroke-width:4px;stroke-linejoin:round;fill:#172b42}
#detail{min-height:26px;overflow-wrap:anywhere}#stage{border-left:3px solid #2463b4;padding:6px 12px;background:#f1f6fc}
.strip{display:flex;gap:3px;margin:7px 0}.strip span{display:block;height:16px;width:24px;border-radius:2px}
table{border-collapse:collapse;width:100%;font-size:13px}th,td{text-align:left;vertical-align:top;border-bottom:1px solid #dce3eb;padding:7px;overflow-wrap:anywhere}
.tablewrap{max-height:180px;overflow:auto}#summary{font-weight:600}input{accent-color:#b84d13}a{color:#245ca5}
.panels{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-top:10px}.panel{min-width:0;border-top:3px solid #2463b4;padding-top:8px;overflow-wrap:anywhere}.panel.b{border-color:#8c54a2}.panel h3{font-size:15px;margin:0 0 6px}.panel table{font-size:12px}.small{font-size:12px;color:#526373}.score{padding:12px;background:#fff5eb;border-left:3px solid #b84d13;margin-top:10px}#pairStatus{font-weight:600}#pair{width:min(100%,700px)}[hidden]{display:none!important}
@media(max-width:650px){body{padding:12px}svg{height:360px}h2{font-size:19px}select{width:100%}.toolbar label{width:100%}}
@media(max-width:650px){.panels{grid-template-columns:1fr}.panel td,.panel th{padding:5px 3px}}
@media(prefers-reduced-motion:reduce){*{scroll-behavior:auto!important}}
</style>
<h2 id="heading"></h2><p id="intro" class="muted"></p><p id="summary" aria-live="polite"></p>
<div class="legend" id="legend"></div>
<div class="toolbar"><label id="nodeLabel">Inspect a node <select id="node" aria-label="Inspect a node"></select></label>
<button id="zoomIn" aria-label="Zoom in">+</button><button id="zoomOut" aria-label="Zoom out">−</button><button id="reset">Reset view</button>
<button id="reveal">Reveal suggested links</button></div>
<div id="pairPicker" class="toolbar" hidden><label>Follow a missing pair <select id="pair" aria-label="Follow a missing pair"></select></label></div>
<div id="walk" hidden><div class="toolbar"><button id="back">Previous</button><button id="next">Next step</button><button id="play">Play</button><span id="stepLabel"></span></div>
<div id="stage" aria-live="polite"></div></div>
<p id="pairStatus" aria-live="polite"></p>
<svg id="graph" viewBox="0 0 1000 550" role="img" aria-label="Interactive graph. Use the node selector for node details.">
<defs><marker id="arrow" markerWidth="7" markerHeight="7" refX="16" refY="3" orient="auto"><path d="M0,0 L0,6 L7,3 z" fill="#2463b4"/></marker>
<marker id="arrowB" markerWidth="7" markerHeight="7" refX="16" refY="3" orient="auto"><path d="M0,0 L0,6 L7,3 z" fill="#8c54a2"/></marker></defs>
<g id="scene"><g id="edges"></g><g id="messages"></g><g id="suggestions"></g><g id="nodes"></g></g></svg>
<p id="detail" aria-live="polite"></p><div id="calculation"></div>
<div id="results"><p id="resultNote"></p><div class="tablewrap"><table><thead><tr><th>Inspect</th><th>Source</th><th>Target</th><th>Original direct link</th></tr></thead><tbody id="rows"></tbody></table></div></div>
<script>
const data=__PAYLOAD__;
const $=id=>document.getElementById(id), svg=$('graph'), ns='http://www.w3.org/2000/svg';
const create=(name,attrs,parent)=>{const el=document.createElementNS(ns,name);Object.entries(attrs).forEach(([k,v])=>el.setAttribute(k,v));parent.appendChild(el);return el};
const element=(tag,text,parent)=>{const el=document.createElement(tag);el.textContent=text;parent.appendChild(el);return el};
const walk=data.mode==='explain', after=data.mode==='after', rounds=data.layers.length, finalStep=2*rounds+2;
$('heading').textContent=walk?'From neighbour messages to one missing link':after?'Reveal the predicted links':'Before GraphSAGE: no predicted links';
$('intro').textContent=walk?'Replay one forward pass of the trained model. Follow the same two nodes from input features to a link score.':'Original links are grey. A path through other nodes is different from a direct link. Node IDs stay fixed in every view.';
$('reveal').hidden=!after;$('walk').hidden=!walk;$('pairPicker').hidden=!walk;$('results').hidden=!after;$('nodeLabel').hidden=walk;
Object.entries(data.types).forEach(([name,color])=>{const label=element('span','',$('legend'));const dot=element('i','',label);dot.className='dot';dot.style.background=color;label.append(document.createTextNode(name))});
['Original link',...((after||walk)?['Predicted link']:[])].forEach((name,i)=>{const l=element('span','',$('legend'));const line=element('i','',l);line.className='line'+(i?' dash':'');l.append(document.createTextNode(name))});
const neighbours=data.nodes.map(()=>[]);data.edges.forEach(([a,b])=>{neighbours[a].push(b);neighbours[b].push(a)});
data.nodes.forEach(n=>{const option=element('option','N'+n.id+' · '+n.type+': '+n.label,$('node'));option.value=n.id});
data.predictions.forEach((r,i)=>{const option=element('option',(i+1)+'. N'+r.source+' '+data.nodes[r.source].label+' → N'+r.target+' '+data.nodes[r.target].label,$('pair'));option.value=i});
let focus=0,step=0,timer=null,scale=1,panX=0,panY=0,pairIndex=0,showLinks=false,inspectOnly=false;
const line=(a,b,attrs,parent)=>create('line',{x1:data.nodes[a].x,y1:data.nodes[a].y,x2:data.nodes[b].x,y2:data.nodes[b].y,...attrs},parent);
data.edges.forEach(([a,b])=>line(a,b,{stroke:'#c0cad5','stroke-width':1.6,'data-edge':a+'-'+b},$('edges')));
const circles=data.nodes.map(n=>{const c=create('circle',{cx:n.x,cy:n.y,r:9,fill:n.color,stroke:'#fff','stroke-width':1.8},$('nodes'));create('title',{},c).textContent='N'+n.id+' · '+n.type+': '+n.label;c.addEventListener('click',()=>{if(!walk){$('node').value=n.id;selectNode()}});create('text',{x:n.x+11,y:n.y+4},$('nodes')).textContent='N'+n.id;return c});
const label=create('text',{},$('nodes')), labelB=create('text',{},$('nodes'));
const fmt=v=>(Math.abs(v)<.0005?0:v).toFixed(3);
function addPrediction(r,i){const el=line(r.source,r.target,{stroke:'#b84d13','stroke-width':4,'stroke-dasharray':'8 5','data-prediction':i},$('suggestions'));create('title',{},el).textContent='New suggestion: N'+r.source+' — N'+r.target;}
function summary(){const count=$('suggestions').childElementCount;$('summary').textContent=data.nodes.length+' nodes · '+data.edges.length+' original links · '+count+' predicted links · '+(data.edges.length+count)+' displayed links';}
function nodeLabel(label,id,prefix){const n=data.nodes[id];label.setAttribute('x',n.x+14);label.setAttribute('y',n.y-17);label.textContent=prefix+n.label.slice(0,30)+(n.label.length>30?'…':'');}
function valueTable(parent,rows){const table=element('table','',parent),head=element('tr','',table);['First 4 values','1','2','3','4'].forEach(t=>element('th',t,head));rows.forEach(([name,values])=>{const row=element('tr','',table);element('th',name,row);values.slice(0,4).forEach(v=>element('td',fmt(v),row))});}
function calculations(r){
 $('calculation').replaceChildren();
 if(!walk)return;
 if(step<=2*rounds){
  const layer=Math.max(0,Math.ceil(step/2)-1),state=data.layers[layer],gather=step%2===1;
  const panels=element('div','',$('calculation'));panels.className='panels';
  [r.source,r.target].forEach((id,j)=>{const panel=element('div','',panels);panel.className='panel'+(j?' b':'');element('h3',(j?'B':'A')+' · N'+id+' '+data.nodes[id].label,panel);element('div','Neighbours: '+neighbours[id].map(n=>'N'+n).join(', '),panel).className='small';
   if(step===0){valueTable(panel,[['Input h⁰',data.vectors[0][id]]]);element('div','Input features encode this node’s name and type.',panel).className='small';}
   else if(gather){valueTable(panel,[['Own input',state.input[id]],['Mean of neighbours',state.mean[id]]]);element('div','Average all direct neighbours’ vectors from the previous layer.',panel).className='small';}
   else{valueTable(panel,[['Self transform',state.self_term[id]],['Neighbour transform + bias',state.neighbour_term[id]],['Updated vector',state.output[id]]]);element('div','Add the two transformed vectors'+(layer<rounds-1?', then apply ReLU.':'; no ReLU in the final layer.'),panel).className='small';}
  });
 }else{
  const box=element('div','',$('calculation'));box.className='score';element('strong','The decoder compares both learned vectors',box);
  valueTable(box,[['Vector A',data.vectors[rounds][r.source]],['Vector B',data.vectors[rounds][r.target]],['A × B by component',r.contributions]]);
  element('p','Sum all '+r.contributions.length+' products: '+fmt(r.score)+' → sigmoid → link score '+fmt(r.link_score),box);
  element('div','Rank '+(pairIndex+1)+' among the displayed absent pairs. This score is not a measured accuracy or a calibrated confidence.',box).className='small';
 }
}
function update(){
 const r=data.predictions[pairIndex],paired=(walk||(after&&!inspectOnly))&&!!r,a=paired?r.source:focus,b=paired?r.target:null;
 const active=new Set([a,b,...neighbours[a],...(b===null?[]:neighbours[b])]);
 circles.forEach((c,i)=>{c.setAttribute('opacity',!walk||active.has(i)?1:.28);c.setAttribute('r',i===a||i===b?13:9);c.setAttribute('stroke',i===a?'#2463b4':i===b?'#8c54a2':'#fff');c.setAttribute('stroke-width',i===a||i===b?4:1.8)});
 nodeLabel(label,a,paired?'A · ':'');labelB.textContent='';if(b!==null)nodeLabel(labelB,b,'B · ');
 $('detail').textContent=paired?'A = N'+a+' '+data.nodes[a].label+' | B = N'+b+' '+data.nodes[b].label:data.nodes[a].type+': '+data.nodes[a].label+' · '+neighbours[a].length+' original neighbours';
 $('pairStatus').textContent=paired?'N'+a+' ↔ N'+b+': original direct link = ABSENT. '+((walk&&step===finalStep)||(after&&showLinks)?'Predicted overlay = visible.':'Predicted overlay = not yet shown.'):'Explore the original links. No predicted links have been added.';
 $('messages').replaceChildren();
 if(walk&&step>0&&step<=2*rounds){[a,b].forEach((id,j)=>neighbours[id].forEach(n=>line(n,id,{stroke:j?'#8c54a2':'#2463b4','stroke-width':2.5,'marker-end':j?'url(#arrowB)':'url(#arrow)','data-message':n+'-'+id},$('messages'))));}
 $('suggestions').replaceChildren();if(walk&&step===finalStep)addPrediction(r,pairIndex);else if(after&&showLinks)data.predictions.forEach(addPrediction);
 if(walk){
  $('stepLabel').textContent='Step '+step+' / '+finalStep;
  const layer=Math.ceil(step/2);
  $('stage').textContent=step===0?'Start: A and B have no direct link. Keep their original neighbour links for message passing.':step===finalStep?'Reveal: draw the selected prediction as a dashed overlay. It is a hypothesis; the original graph stays unchanged.':step===finalStep-1?'Decode: multiply matching components of the two final vectors, sum them, then apply sigmoid.':step%2?'Layer '+layer+' — gather: messages arrive from each node’s direct neighbours'+(layer>1?', carrying information already gathered in earlier layers.':'.'):'Layer '+layer+' — update: transform the node’s own vector and its neighbour mean with learned weights, then combine them.';
  $('back').disabled=step===0;$('next').disabled=step===finalStep;
  calculations(r);
 }
 summary();
}
function stop(){if(timer!==null)clearInterval(timer);timer=null;$('play').textContent='Play';}
function selectNode(){stop();focus=Number($('node').value);inspectOnly=true;update();}
$('node').addEventListener('change',selectNode);
$('pair').onchange=()=>{stop();pairIndex=Number($('pair').value);step=0;update()};
$('back').onclick=()=>{stop();step=Math.max(0,step-1);update()};
$('next').onclick=()=>{stop();step=Math.min(finalStep,step+1);update()};
$('play').onclick=()=>{if(timer!==null){stop();return;}step=0;update();$('play').textContent='Pause';timer=setInterval(()=>{step++;update();if(step===finalStep)stop()},2400)};
if(matchMedia('(prefers-reduced-motion: reduce)').matches){$('play').hidden=true;}
function transform(){$('scene').setAttribute('transform','translate('+panX+' '+panY+') translate(500 275) scale('+scale+') translate(-500 -275)');}
$('zoomIn').onclick=()=>{scale=Math.min(4,scale*1.25);transform()};$('zoomOut').onclick=()=>{scale=Math.max(.5,scale/1.25);transform()};
$('reset').onclick=()=>{scale=1;panX=panY=0;transform()};
let drag=null;svg.addEventListener('pointerdown',e=>{if(e.target.tagName==='circle')return;drag={x:e.clientX,y:e.clientY,px:panX,py:panY};svg.setPointerCapture(e.pointerId)});
svg.addEventListener('pointermove',e=>{if(!drag)return;const ratio=1000/svg.getBoundingClientRect().width;panX=drag.px+(e.clientX-drag.x)*ratio;panY=drag.py+(e.clientY-drag.y)*ratio;transform()});
svg.addEventListener('pointerup',()=>drag=null);svg.addEventListener('pointercancel',()=>drag=null);
$('reveal').onclick=()=>{showLinks=!showLinks;$('reveal').textContent=showLinks?'Hide suggested links':'Reveal suggested links';update()};
if(after){
 $('resultNote').textContent=data.pairLabel+' · requested '+data.requested+', found '+data.predictions.length+'. Dashed links are model suggestions to investigate, not established facts.';
 if(!data.predictions.length)$('resultNote').textContent+=' No new links to show for this setting. Choose another type pair or increase the requested count.';
 data.predictions.forEach((r,i)=>{const row=element('tr','',$('rows')),cell=element('td','',row),button=element('button','Pair '+(i+1),cell);button.onclick=()=>{pairIndex=i;focus=r.source;$('node').value=r.source;inspectOnly=false;update()};['N'+r.source+' '+data.nodes[r.source].label,'N'+r.target+' '+data.nodes[r.target].label,'Absent'].forEach(t=>element('td',t,row))});
}
update();
</script></html>'''

_CLASS_GRAPH = {'nodes': [{'id': 'Material::FEC', 'label': 'Fluoroethylene carbonate', 'type': 'Material'}, {'id': 'Cell::li%7C%7Ccu%20cells', 'label': 'Li||Cu cells', 'type': 'Cell'}, {'id': 'Electrolyte::pafp%20electrolyte', 'label': 'PAFP electrolyte', 'type': 'Electrolyte'}, {'id': 'Property::cycling%20performance', 'label': 'cycling performance', 'type': 'Property'}, {'id': 'Process::modified%20aurbach%20method', 'label': 'modified Aurbach method', 'type': 'Process'}, {'id': 'Condition::25%20%C2%B0c', 'label': '25 °C', 'type': 'Condition'}, {'id': 'Material::EC', 'label': 'EC', 'type': 'Material'}, {'id': 'Cell::li%7C%7Cli%20symmetric%20cell', 'label': 'Li||Li symmetric cell', 'type': 'Cell'}, {'id': 'Process::galvanostatic%20cycling', 'label': 'Galvanostatic cycling', 'type': 'Process'}, {'id': 'Condition::room%20temperature', 'label': 'room temperature', 'type': 'Condition'}, {'id': 'Electrolyte::te-i', 'label': 'TE-I', 'type': 'Electrolyte'}, {'id': 'Property::ionic%20conductivity', 'label': 'ionic conductivity', 'type': 'Property'}, {'id': 'Cell::li%7C%7Cncm811%20coin%20cell', 'label': 'Li||NCM811 coin cell', 'type': 'Cell'}, {'id': 'Material::LiTFSI', 'label': 'LiTFSI', 'type': 'Material'}, {'id': 'Electrolyte::litfsi-peo-0.2alf3', 'label': 'LiTFSI-PEO-0.2AlF3', 'type': 'Electrolyte'}, {'id': 'Property::capacity%20retention', 'label': 'capacity retention', 'type': 'Property'}, {'id': 'Process::cyclic%20voltammetry', 'label': 'cyclic voltammetry', 'type': 'Process'}, {'id': 'Condition::90%20%C2%B0c', 'label': '90 °C', 'type': 'Condition'}, {'id': 'Material::li%20metal', 'label': 'Li metal', 'type': 'Material'}, {'id': 'Cell::li%7C%7Ccu%20cell', 'label': 'Li||Cu cell', 'type': 'Cell'}], 'edges': [{'source': 'Material::FEC', 'target': 'Cell::li%7C%7Ccu%20cells'}, {'source': 'Material::FEC', 'target': 'Electrolyte::pafp%20electrolyte'}, {'source': 'Material::FEC', 'target': 'Property::cycling%20performance'}, {'source': 'Cell::li%7C%7Ccu%20cells', 'target': 'Process::modified%20aurbach%20method'}, {'source': 'Cell::li%7C%7Ccu%20cells', 'target': 'Material::EC'}, {'source': 'Electrolyte::pafp%20electrolyte', 'target': 'Cell::li%7C%7Cli%20symmetric%20cell'}, {'source': 'Electrolyte::pafp%20electrolyte', 'target': 'Cell::li%7C%7Ccu%20cell'}, {'source': 'Process::modified%20aurbach%20method', 'target': 'Condition::25%20%C2%B0c'}, {'source': 'Condition::25%20%C2%B0c', 'target': 'Process::galvanostatic%20cycling'}, {'source': 'Condition::25%20%C2%B0c', 'target': 'Property::ionic%20conductivity'}, {'source': 'Condition::25%20%C2%B0c', 'target': 'Process::cyclic%20voltammetry'}, {'source': 'Material::EC', 'target': 'Electrolyte::te-i'}, {'source': 'Cell::li%7C%7Cli%20symmetric%20cell', 'target': 'Material::li%20metal'}, {'source': 'Process::galvanostatic%20cycling', 'target': 'Condition::room%20temperature'}, {'source': 'Process::galvanostatic%20cycling', 'target': 'Cell::li%7C%7Cncm811%20coin%20cell'}, {'source': 'Process::galvanostatic%20cycling', 'target': 'Condition::90%20%C2%B0c'}, {'source': 'Process::galvanostatic%20cycling', 'target': 'Cell::li%7C%7Ccu%20cell'}, {'source': 'Electrolyte::te-i', 'target': 'Material::LiTFSI'}, {'source': 'Property::ionic%20conductivity', 'target': 'Condition::90%20%C2%B0c'}, {'source': 'Cell::li%7C%7Cncm811%20coin%20cell', 'target': 'Property::capacity%20retention'}, {'source': 'Material::LiTFSI', 'target': 'Electrolyte::litfsi-peo-0.2alf3'}, {'source': 'Material::li%20metal', 'target': 'Cell::li%7C%7Ccu%20cell'}]}
lab = GraphSAGELab(graph=_CLASS_GRAPH)
print('Ready. Run the next cell to view the graph.')

## 2. Before GraphSAGE: explore the graph
This viewer shows **20 connected nodes from the class graph**, with colours for node types and **zero predicted links**.
Select a node to read its name and inspect its neighbours. The same nodes and positions stay in every viewer.

For this exercise, links are undirected and relationship labels are ignored.

In [ ]:
lab.before()

## 3. Train GraphSAGE
Learn node representations from the existing links and sampled unconnected pairs. Training leaves the graph unchanged.

In [ ]:
lab.fit(layers=2)
print('Model ready. No links have been added. Follow one prediction in the next cell.')

## 4. Follow one missing link: messages → vectors → score
Choose a suggested pair, then press **Next step** or **Play**.

**No direct link → gather neighbours → update both nodes → repeat → calculate a link score → reveal the suggestion.**

The viewer replays the trained model's actual calculations. Blue arrows carry information along existing links;
the orange predicted link appears only at the final step. Message passing updates vectors; the decoder scores a possible new link.
The same node IDs and positions stay fixed throughout.

In [ ]:
lab.explain_link()

## 5. After GraphSAGE: see the suggested links
The model ranks **three Material–Property pairs with no existing direct link**. Existing links stay grey.
Press **Reveal suggested links** to add the orange dashed suggestions. Select a result row to identify both endpoints.

The dashed links are hypotheses to investigate. This exercise explores suggestions without measuring prediction accuracy.

In [ ]:
lab.after(source_type='Material', target_type='Property', top_k=3)

**Main exercise complete.** Select one suggested connection and describe what changed in the graph.

---

## Optional · Change the graph you connect
Change **one setting at a time**, rerun the cell, and compare the viewers.

| Setting | Try | What changes |
|---|---|---|
| `LAYERS` | `1`, `2`, `3` | How many rounds of neighbour information the model uses |
| `SOURCE_TYPE`, `TARGET_TYPE` | `Material`, `Property`, `Process`, `Condition`, `Electrolyte`, `Cell` | Which node types may receive suggested connections |
| `NEW_LINKS` | `1`, `3`, `5` | Maximum number of new links to display |

**Try:** first increase `NEW_LINKS` to `5`. Then keep that count and change `TARGET_TYPE` to `'Process'`.
Finally compare `LAYERS = 1` and `LAYERS = 2` with the same type pair and count.
The viewer reports the actual number found; some settings have fewer available pairs.
Every run starts from the same original graph, so suggestions do not accumulate.

In [ ]:
LAYERS = 2
SOURCE_TYPE, TARGET_TYPE = 'Material', 'Property'
NEW_LINKS = 5

lab.fit(layers=LAYERS)
lab.after(source_type=SOURCE_TYPE, target_type=TARGET_TYPE, top_k=NEW_LINKS)